# Task 03 — Gold: Daily Revenue Aggregation — **SOLUTION**

**Workshop: Final Pipeline | Layer 3 of 3**

## Goal

Read `silver.lab_orders`, aggregate to daily metrics, write to `gold.lab_daily_orders`.

```
silver.lab_orders
        |
        v  to_date * groupBy * agg
        v
gold.lab_daily_orders
```

## Expected output schema

| Column | Type | Description |
|--------|------|-------------|
| order_date | DateType | Calendar day from `order_datetime` |
| order_count | LongType | Total orders that day |
| total_revenue | DoubleType | SUM of `net_amount` |
| avg_order_value | DoubleType | AVG of `net_amount`, rounded to 2 dp |

> Use `net_amount` (after discount), not `total_amount`. One row = one calendar day.

In [0]:
%run ../../../setup/00_setup

In [0]:
dbutils.widgets.text("catalog",       CATALOG,       "Catalog")
dbutils.widgets.text("silver_schema", SILVER_SCHEMA, "Silver Schema")
dbutils.widgets.text("gold_schema",   GOLD_SCHEMA,   "Gold Schema")

catalog       = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema   = dbutils.widgets.get("gold_schema")

source_table = f"{catalog}.{silver_schema}.lab_orders"
target_table = f"{catalog}.{gold_schema}.lab_daily_orders"

print(f"Source : {source_table}")
print(f"Target : {target_table}")

---

## Exercise 1 — Imports

Import aggregate functions and `to_date` from `pyspark.sql.functions`.

> Note: Python has built-in `sum()`, `round()`, and `min()`.
> Importing the same names from `pyspark.sql.functions` shadows them here — that is intentional.

In [0]:
from pyspark.sql.functions import to_date, col, count, sum, avg, round

---

## Exercise 2 — Read the Silver table

In [0]:
# Read the Silver table into orders_silver (Task 02 must have run first)
orders_silver = spark.table(source_table)

In [0]:
print(f"Silver rows : {orders_silver.count():,}")
orders_silver.printSchema()

---

## Exercise 3 — Extract `order_date` from `order_datetime`

Add a new column `order_date` (DateType) by extracting the calendar date
from `order_datetime` (TimestampType).

In [0]:
# Add order_date (DateType) by extracting the date from order_datetime (TimestampType)
orders_with_date = (
    orders_silver
    .withColumn("order_date", to_date(col("order_datetime")))
)

---

## Exercise 4 — Aggregate: one row per day

Group by `order_date`, compute the four metrics, sort ascending by date.
Name the result `daily_df`.

In [0]:
# Group by order_date, compute:
#   order_count     = count("*")
#   total_revenue   = sum("net_amount")
#   avg_order_value = round(avg("net_amount"), 2)
# Order the result by order_date ascending
daily_df = (
    orders_with_date
    .groupBy("order_date")
    .agg(
        count("*").alias("order_count"),
        sum("net_amount").alias("total_revenue"),
        round(avg("net_amount"), 2).alias("avg_order_value")
    )
    .orderBy("order_date")
)

In [0]:
print(f"Daily rows : {daily_df.count():,}  (= distinct order dates)")
daily_df.show(15, truncate=False)

---

## Exercise 5 — Write to Gold as a managed Delta table

In [0]:
# Write daily_df to target_table as a managed Delta table
# mode: overwrite (Gold is always fully rebuilt from Silver)
# No overwriteSchema needed — Gold schema is stable
(
    daily_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

In [0]:
row_count = spark.table(target_table).count()
print(f"OK  {target_table}  ->  {row_count:,} daily records")
display(spark.table(target_table))

In [0]:
import json
dbutils.notebook.exit(json.dumps({"status":"SUCCESS","table":target_table,"rows":row_count}))